In [ ]:
!nvidia-smi

Sun Aug  4 03:06:27 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
!pip3 install -q -U bitsandbytes==0.42.0
!pip3 install -q -U peft==0.8.2
!pip3 install -q -U trl==0.7.10
!pip3 install -q -U dataset==2.17.0
!pip3 install -q -U transformers==4.38.0
!pip3 install -q -U accelerate==0.27.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 761.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 622.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 672.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 957.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from huggingface_hub import login
login()

In [ ]:

model_id ="google/gemma-2b-it"

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model=AutoModelForCausalLM.from_pretrained(model_id,quantization_config=bnb_config,device_map={"":0})
tokenizer=AutoTokenizer.from_pretrained(model_id, add_eos_token=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
def get_completion(query: str,model,tokenizer)->str:
  device="cuda:0"

  prompt_template="""
  <start_of_turn>user
  Below is an instruction that describes a task. Write a response that appropriately completes the request.
  {query}
  <end_of_turn>\n<start_of_turn>model
  """

  prompt=prompt_template.format(query=query)
  encodes = tokenizer(prompt,return_tensors="pt",add_special_tokens=True)
  model_inputs=encodes.to(device)

  generated_ids = model.generate(**model_inputs,max_new_tokens=1000, do_sample=True, pad_token_id=tokenizer.eos_token_id)
  decoded =tokenizer.decode(generated_ids[0],skip_special_tokens=True)
  return (decoded)

In [ ]:
result=get_completion(query="genrate a story using genre as fantasy", model=model,tokenizer=tokenizer)
print(result)

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



  user
  Below is an instruction that describes a task. Write a response that appropriately completes the request.
  genrate a story using genre as fantasy
  
model
  Generated Story using Genre as Fantasy

In the verdant realm of Eldoria, where soaring mountains pierced the heavens and rolling plains stretched as far as the eye could see, resided an extraordinary sorceress named Lyra. With hair as golden as the sunlit sands of the desert and eyes that mirrored the shimmering waters of the Great Lake Cadence, Lyra possessed a heart aflame with a passion for adventure and a deep love for storytelling.

One fateful day, as Lyra sat on the banks of the River Lumina, her gaze swept over the enchanted forests that whispered secrets in the breeze. The enchanting melody of a nearby melody caught her ears, and she knew that she had stumbled upon a realm of her own.

With a magical flick of her fingers, Lyra summoned a magnificent unicorn, its tail shimmering like a thousand stars upon the nig

In [ ]:
result=get_completion(query="code the fibonacci series in python using recursion", model=model,tokenizer=tokenizer)
print(result)

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



  user
  Below is an instruction that describes a task. Write a response that appropriately completes the request.
  code the fibonacci series in python using recursion
  
model
  ```python
def fibonacci(n):
    if n == 0:
      return 0
    elif n == 1:
      return 1
    else:
      return fibonacci(n-1) + fibonacci(n-2)

# Print the first 10 numbers in the Fibonacci sequence.
for i in range(10):
    print(fibonacci(i))
```


In [ ]:
from datasets import load_dataset
dataset=load_dataset("AtlasUnified/atlas-storyteller", split="train")
dataset

Generating train split:   0%|          | 0/5018 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'Story'],
    num_rows: 5018
})

In [ ]:
def generate_prompt(data_point):
  """Gen. input text based on a prompt and story.
  :param data_point:dict:Data point
  :return:dict: tokenized prompt
  """
  prefix_text='Below is a story. Write a summary of the story.\n\n'
  story = data_point.get("story", "")
  text = f"""<start_of_turn>user {prefix_text}{story}<end_of_turn>\n<start_of_turn>model """
  return text

text_column = [generate_prompt(data_point) for data_point in dataset]
dataset =dataset.add_column("prompt",text_column)

In [ ]:
def generate_prompt(data_point):
  """Gen. input text based on a prompt, task instruction, (context info.), and answer
  :param data_point:dict:Data point
  :return:dict: tokenized prompt
  """
  prefix_text='Below is an instruction that describes a task. Write a response that '\
            'appropriately completes the request.\n\n'
  if data_point["input"]:
    text = f"""<start_of_turn>user {prefix_text}{data_point["instruction"]}here are the inputs {data_point["input"]}<end_of_turn>\n<start_of_turn>model {data_point["output"]}<end_of_turn>"""
  else:
    text = f"""<start_of_turn>user {prefix_text}{data_point["instruction"]}<end_of_turn>\n<start_of_turn>model {data_point["output"]}<end_of_turn>"""
  return text

text_column = [generate_prompt(data_point) for data_point in dataset]
dataset =dataset.add_column("prompt",text_column)

KeyError: 'input'

In [ ]:
# def generate_prompt(data_point):
#   """Gen. input text based on genre and title to generate a new story.
#   :param data_point:dict:Data point
#   :return:dict: tokenized prompt
#   """
#   prefix_text='Below is an instruction that describes a task. Write a response that '\
#             'appropriately completes the request.\n\n'
#   if "genre" in data_point and "title" in data_point:
#     text = f"""<start_of_turn>user {prefix_text}Generate a new story in the {data_point["genre"]} genre with the title '{data_point["title"]}'<end_of_turn>\n<start_of_turn>model """
#     return text
#   else:
#     # Handle the case where keys are missing
#     return ""

# text_column = [generate_prompt(data_point) for data_point in dataset]
# dataset =dataset.add_column("prompt",text_column)

In [ ]:
dataset =dataset.shuffle(seed=1234)

In [ ]:
dataset =dataset.train_test_split(test_size=0.2)

train_data =dataset["train"]
test_data =dataset["test"]

In [ ]:
from peft import LoraConfig,get_peft_model,PeftModel,prepare_model_for_kbit_training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
print(model)

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
     

In [ ]:
import bitsandbytes as bnb
def find_all_linear_names(model):
  cls=bnb.nn.Linear4bit
  lora_module_names = set()
  for name, module in model.named_modules():
    if isinstance(module,cls):
      names=name.split(".")
      lora_module_names.add(names[0] if len(names)==1 else names[-1])
      if "lm_head" in lora_module_names:
        lora_module_names.remove("lm_head")
  return list(lora_module_names)

modules=find_all_linear_names(model)
print(modules)

['o_proj', 'up_proj', 'down_proj', 'q_proj', 'gate_proj', 'v_proj', 'k_proj']


In [ ]:
from peft import LoraConfig,get_peft_model
lora_config=LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)

In [ ]:
trainable, total =model.get_nb_trainable_parameters()
print(f"Trainable: {trainable}| total: {total}| Percentage:{trainable/total*100:.4f}%")

Trainable: 78446592| total: 2584619008| Percentage:3.0351%


In [ ]:
import transformers
from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=test_data,
    peft_config=lora_config,
    dataset_text_field="prompt",
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=0.03,
        # num_train_epochs=1,
        max_steps=10,
        learning_rate=2e-4,
        # fp16=True
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit",
        save_strategy="epoch",
    ),

    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer,mlm=False)
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:223: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/4014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1004 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:290: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


In [ ]:
model.config.use_cache=False
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
1,12.081600
2,12.081600
3,9.193800
4,7.115800
5,5.695100
6,4.792600
7,4.026400
8,3.683600
9,3.425300
10,3.249900


TrainOutput(global_step=10, training_loss=6.534558129310608, metrics={'train_runtime': 31.2363, 'train_samples_per_second': 1.281, 'train_steps_per_second': 0.32, 'total_flos': 10384068280320.0, 'train_loss': 6.534558129310608, 'epoch': 0.01})

In [ ]:
new_model="Story-generator-gemma-instruct1"
trainer.model.save_pretrained(new_model)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import gc
gc.collect()


139

In [ ]:
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:

base_model =AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"":0}
)
merged_model =PeftModel.from_pretrained(base_model,new_model)
merged_model =merged_model.merge_and_unload()

merged_model.save_pretrained("merged_model",safe_serialization=True)

# tokenizer =AutoTokenizer.from_pretrained("model_id")
tokenizer.padding_side="right"
tokenizer.save_pretrained("merged_model")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

('merged_model/tokenizer_config.json',
 'merged_model/special_tokens_map.json',
 'merged_model/tokenizer.model',
 'merged_model/added_tokens.json',
 'merged_model/tokenizer.json')

In [ ]:
from huggingface_hub import login
import os

# Set your Hugging Face token as an environment variable


# Login to Hugging Face Hub
login(os.getenv('HF_TOKEN'))

In [ ]:
# merged_model.push_to_hub(new_model,use_temp_dir=False)
# tokenizer.push_to_hub(new_model,use_temp_dir=False)
merged_model.push_to_hub(new_model, use_temp_dir=False, token=os.getenv('HF_TOKEN'))
print("Merged model pushed to Hugging Face Hub.")

print("Pushing tokenizer to Hugging Face Hub...")
tokenizer.push_to_hub(new_model, use_temp_dir=False, token=os.getenv('HF_TOKEN'))

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Merged model pushed to Hugging Face Hub.
Pushing tokenizer to Hugging Face Hub...


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/klvnakhila/Story-generator-gemma-instruct1/commit/e29a17a0db3d392d96f826247d0980ae1560c3e4', commit_message='Upload tokenizer', commit_description='', oid='e29a17a0db3d392d96f826247d0980ae1560c3e4', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
# Import or define the get_completion function.
# For example, if it's part of the transformers library:
from transformers import pipeline

# Assuming you want to use text generation pipeline
get_completion = pipeline('text-generation', model='klvnakhila/Story-generator-gemma-instruct1')

# Pass the text input as a positional argument
result = get_completion("Captain Samuel Grant stood tall on the deck of his ship, the HMS Valiant, as it sailed through the choppy waters of the Atlantic Ocean. The year was 1798, and tensions were high between Britain and France.", max_length=100)
print(result)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
